# 03 — End-to-End Pipeline Demo

Demonstrates the full Zero-Trust Deepfake Voice Defense pipeline:
1. Audio loading and preprocessing
2. CNN + Whisper forensic analysis
3. Trust score aggregation
4. Decision (pass / challenge / reject)
5. Dynamic liveness challenge generation


In [ ]:
import sys
sys.path.insert(0, '..')

from src.decision.threshold_engine import ThresholdEngine
from src.decision.trust_scorer import TrustScorer
from src.decision.action_router import ActionRouter
from src.liveness.challenge_generator import ChallengeGenerator
from src.utils.config_loader import load_config

threshold_cfg = load_config('../configs/thresholds_config.yaml')
print('Thresholds loaded:', threshold_cfg.get('thresholds', {}))


In [ ]:
# Simulate a pipeline run with mock scores
scorer = TrustScorer()
engine = ThresholdEngine()
router = ActionRouter()
gen = ChallengeGenerator()

# Scenario 1: Genuine audio (high scores)
trust = scorer.score(cnn_score=0.92, whisper_score=0.88, liveness_passed=None)
decision = engine.evaluate(trust)
action = router.route(decision)
print(f'Scenario 1 — trust={trust:.3f}, decision={decision}, action={action}')

# Scenario 2: Uncertain audio → challenge
trust = scorer.score(cnn_score=0.62, whisper_score=0.55, liveness_passed=None)
decision = engine.evaluate(trust)
if decision == 'challenge':
    challenge = gen.generate()
    print(f'Scenario 2 — challenge issued: "{challenge}"')

# Scenario 3: Synthetic audio (low scores)
trust = scorer.score(cnn_score=0.10, whisper_score=0.15, liveness_passed=False)
decision = engine.evaluate(trust)
action = router.route(decision)
print(f'Scenario 3 — trust={trust:.3f}, decision={decision}, action={action}')
